In [1]:
import os
import torch 
import ollama
import re
import tempfile as tem
import subprocess as sp
import inspect
from ollama import  chat,ChatResponse
from pprint import pprint
# from IPython.display import display,Image
# from langgraph.graph import StateGraph,START,END
# from langchain.tools import tool
# from langchain_ollama import ChatOllama
# from langchain.agents import create_agent
# from langchain.messages import HumanMessage,SystemMessage
# from langfuse import Langfuse,observe

In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

In [3]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

I have access to information up until my last update in October 2023. My capabilities in Verilog, like in any technical field, can be considered strong based on my training data but may not be perfect as I don't have real-time coding experience or the ability to execute code. If you have specific questions or need help with Verilog, feel free to ask!


In [4]:
def code_gen(prompt:str)->str:
    stream=chat(model=Ollama_MODEL,messages=[{'role':'System',
                                           "content":"You are a Senoir Design Engineer and well versed with RTL code, provide a clean,synthesizable,well commented,compact Verilog code with respect to the user request."},
                                           {"role":'user',
                                            "content":f"{prompt}"}],
                                            stream=True)
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [5]:
User_inpt=input("Prompt the verilog moduel requesting")
response=code_gen(User_inpt)

Sure! Below is a clean, synthesizable, and well-commented Verilog code for an APB (Advanced Peripheral Bus) completion module. This module will handle the completion of APB transactions by managing the acknowledge signals and updating the data registers.

```verilog
module apb_completer (
    input wire clk,          // Clock signal
    input wire rst_n,        // Active low reset

    // APB Slave Interface
    input wire [31:0] psel,   // Peripheral Select
    input wire penable,      // Peripheral Enable
    input wire [31:0] paddr,  // Peripheral Address
    input wire [31:0] pwdata, // Write Data
    output reg [31:0] prdata,// Read Data
    input wire pwrite,       // Write enable
    input wire pready,       // Peripheral ready (output)
    output reg penack,       // Peripheral acknowledge

    // APB Master Interface
    output reg msel,         // Master Select
    output reg menable,      // Master Enable
    input wire [31:0] mrdata, // Master Read Data
    output reg [31:0

In [6]:
print(response)

Sure! Below is a clean, synthesizable, and well-commented Verilog code for an APB (Advanced Peripheral Bus) completion module. This module will handle the completion of APB transactions by managing the acknowledge signals and updating the data registers.

```verilog
module apb_completer (
    input wire clk,          // Clock signal
    input wire rst_n,        // Active low reset

    // APB Slave Interface
    input wire [31:0] psel,   // Peripheral Select
    input wire penable,      // Peripheral Enable
    input wire [31:0] paddr,  // Peripheral Address
    input wire [31:0] pwdata, // Write Data
    output reg [31:0] prdata,// Read Data
    input wire pwrite,       // Write enable
    input wire pready,       // Peripheral ready (output)
    output reg penack,       // Peripheral acknowledge

    // APB Master Interface
    output reg msel,         // Master Select
    output reg menable,      // Master Enable
    input wire [31:0] mrdata, // Master Read Data
    output reg [31:0

In [7]:
def code_extraction(code:str)->str:
    pattern=r'module\s+\w+.*?endmodule'
    matches=re.findall(pattern=pattern,string=code,flags=re.DOTALL)
    clean_code=matches[0]
    return clean_code

In [8]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module will handle the completion of APB transactions by managing the acknowledge signals and updating the data registers.

```verilog
module apb_completer (
    input wire clk,          // Clock signal
    input wire rst_n,        // Active low reset

    // APB Slave Interface
    input wire [31:0] psel,   // Peripheral Select
    input wire penable,      // Peripheral Enable
    input wire [31:0] paddr,  // Peripheral Address
    input wire [31:0] pwdata, // Write Data
    output reg [31:0] prdata,// Read Data
    input wire pwrite,       // Write enable
    input wire pready,       // Peripheral ready (output)
    output reg penack,       // Peripheral acknowledge

    // APB Master Interface
    output reg msel,         // Master Select
    output reg menable,      // Master Enable
    input wire [31:0] mrdata, // Master Read Data
    output reg [31:0] mwdata, // Master Write Data
    output reg mwrite,       // Master write enable
    output reg mready,       // Master ready (inp

In [13]:
def linter(code:str)->dict:
    #temp file creation
     with tem.NamedTemporaryFile(mode="w+t",suffix=".v",delete=False) as temp_file:
          temp_file.write(code)
          temp_file_path=temp_file.name
          try:     
               cmd=["verilator",
                    "--lint-only",
                    "-Wall",
                    "-Wno-UNUSED",
                    "-Wno-WIDTH",
               temp_file_path]
               out=sp.run(cmd,capture_output=True,text=True)
               if out.returncode!=0:
                    return{
                         "status":"error",
                         "return_code":out.returncode,
                         "stderr":out.stderr
                    }
               match=re.search(r"\bmodule\s+(\w+)",code)
               if not match:
                    return{
                              "status": "error",
                              "returncode": out.returncode,
                              "stderr": "No Verilog module found." 
                         }
               top_module=match.group(1)
               output_file=os.path.join(os.getcwd(),
                                        f"{top_module}.v")
               with open(output_file,"w") as f:
                    f.write(code)
          except sp.CalledProcessError as e:
               return e.stderr
          finally:
               if os.path.exists(temp_file_path):
                    os.remove(temp_file_path)
     return {"stdout":out.stdout,"return_code":out.returncode}

In [14]:
res=linter(response)
print(res)

{'status': 'error', 'return_code': 1, 'stderr': '%Error: No top level module found\n%Error: Exiting due to 1 error(s)\n'}
